# Step 7 — Transfer Learning for Molecular Property Prediction

## Strategy

**Why transfer learning?**  
Our dataset has 1,571 molecules — too small to train a neural network from scratch (R² = −5.41 in Step 3).  
Transfer learning reuses a neural network **already trained on millions of molecules** from public databases,
then fine-tunes only the final layers on our 1,571 molecules.

This is the same idea as using ImageNet-pretrained ResNet for a small image dataset.

## Two approaches in this notebook

| Approach | Pretrained source | What we fine-tune |
|----------|-------------------|-------------------|
| **A — ChemBERTa embeddings** | HuggingFace `seyonec/ChemBERTa-zinc-base-v1` (77M params, pretrained on 100k ZINC SMILES) | Freeze transformer, train a regression head on top |
| **B — Pretrained descriptor NN** | MoleculeNet ESOL/FreeSolv task weights (via DeepChem) | Replace output head, fine-tune last 2 layers only |

**Baseline we are trying to beat:** Random Forest R² ≈ 0.659 (HOMO), 0.385 (LUMO), 0.642 (Bandgap)

---

> **Colab tip:** Set Runtime → Change runtime type → **T4 GPU** before running.

In [ ]:
# ── Cell 1: Install dependencies ──────────────────────────────────────────
# ChemBERTa (transformers + tokenizers) and DeepChem for pretrained models
%pip install -q transformers tokenizers torch torchvision
%pip install -q deepchem
%pip install -q rdkit pandas numpy scikit-learn matplotlib seaborn

print('✓ All packages installed.')
print('If this is your first run, RESTART the runtime now, then skip back to Cell 2.')

In [ ]:
# ── Cell 2: Imports & setup ────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

from transformers import AutoTokenizer, AutoModel
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    print('⚠  No GPU detected — ChemBERTa fine-tuning will be slow. Use Colab T4 GPU.')

plt.rcParams.update({
    'figure.facecolor': 'white', 'axes.facecolor': 'white',
    'axes.grid': True, 'grid.alpha': 0.3, 'font.size': 12,
})
print('✓ Imports complete.')

In [ ]:
# ── Cell 3: Load data (same as all previous steps) ────────────────────────
from pathlib import Path

def first_existing(paths):
    for p in paths:
        if Path(p).exists():
            return Path(p)
    return None

candidate_roots = [
    Path.cwd(),
    Path('/content/drive/MyDrive/molecular-property-prediction'),
    Path('/content/drive/MyDrive/Colab Notebooks/molecular-property-prediction'),
    Path('/content/drive/MyDrive'),
    Path('/content'),
]

data_source = None
fp_source   = None
rf_source   = None

for root in candidate_roots:
    data_source = first_existing([
        root / 'Data_Final_merged.xlsx',
        root / 'Data_Final_merged.csv',
        root / 'Each_Step_Output_Download/Datasets/Data_Final_merged.xlsx',
        root / 'Each_Step_Output_Download/Datasets/Data_Final_merged.csv',
    ])
    fp_source = first_existing([
        root / 'features_morgan.npy',
        root / 'Each_Step_Output_Download/Morgan&Maccs/features_morgan.npy',
    ])
    rf_source = first_existing([
        root / 'models_rf.pkl',
        root / 'Each_Step_Output_Download/Models_saved/models_rf.pkl',
    ])
    if data_source and fp_source:
        break

if not data_source:
    try:
        from google.colab import files
        print('Files not found on Drive. Please upload:')
        print('  1. Data_Final_merged.xlsx (or .csv)')
        print('  2. features_morgan.npy')
        print('  3. models_rf.pkl  (optional — for baseline comparison)')
        uploaded = files.upload()
        data_source = first_existing([Path('Data_Final_merged.xlsx'), Path('Data_Final_merged.csv')])
        fp_source   = first_existing([Path('features_morgan.npy')])
        rf_source   = first_existing([Path('models_rf.pkl')])
    except Exception:
        pass

# Load
if data_source.suffix.lower() == '.csv':
    df = pd.read_csv(data_source)
else:
    df = pd.read_excel(data_source)

X_morgan = np.load(fp_source)
smiles_list = df['SMILES_acc'].tolist()

y_homo = df['HOMO_A'].values.astype(np.float32)
y_lumo = df['LUMO_A'].values.astype(np.float32)
y_eg   = df['EgA_opt'].values.astype(np.float32)

print(f'✓ Dataset loaded: {df.shape}')
print(f'  Molecules: {len(smiles_list)}')
print(f'  Morgan FP shape: {X_morgan.shape}')

# Load RF baseline if available
rf_models = None
if rf_source:
    with open(rf_source, 'rb') as f:
        rf_models = pickle.load(f)
    print('✓ RF models loaded for baseline comparison')
else:
    print('⚠  RF model not found — will re-train RF baseline in Cell 6')

In [ ]:
# ── Cell 4: Train/test split — same random_state=42 as all previous steps ─
indices = np.arange(len(smiles_list))

idx_train, idx_test = train_test_split(indices, test_size=0.2, random_state=42)

# SMILES subsets
smiles_train = [smiles_list[i] for i in idx_train]
smiles_test  = [smiles_list[i] for i in idx_test]

# Morgan FP subsets (for baseline comparison)
X_train_fp = X_morgan[idx_train]
X_test_fp  = X_morgan[idx_test]

# Target subsets
targets = {'HOMO': y_homo, 'LUMO': y_lumo, 'Eg': y_eg}
y_train = {k: v[idx_train] for k, v in targets.items()}
y_test  = {k: v[idx_test]  for k, v in targets.items()}

print(f'Train: {len(idx_train)} | Test: {len(idx_test)}')
print('Split matches Step 3 (random_state=42) ✓')

---
## Approach A — ChemBERTa Transfer Learning

**Model:** `seyonec/ChemBERTa-zinc-base-v1`  
- RoBERTa transformer pretrained on 100,000 SMILES strings from ZINC
- Takes raw SMILES text as input (no fingerprints needed)
- 768-dimensional molecular embedding from `[CLS]` token

**Strategy:**
1. **Frozen embeddings** — extract 768-d vectors, train RF on top (fast baseline)
2. **Fine-tuned** — unfreeze last 2 transformer layers, train end-to-end with regression head

**Why this helps:** The model has already learned chemistry from 100k molecules.  
Our 1,571 molecules just need to teach it the *specific* HOMO/LUMO/Bandgap mapping.

In [ ]:
# ── Cell 5: Load ChemBERTa & extract frozen embeddings ────────────────────
MODEL_NAME = 'seyonec/ChemBERTa-zinc-base-v1'

print(f'Loading {MODEL_NAME}...')
print('(~500 MB download on first run)')

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
chemberta  = AutoModel.from_pretrained(MODEL_NAME)
chemberta  = chemberta.to(DEVICE)
chemberta.eval()

print(f'✓ ChemBERTa loaded')
print(f'  Parameters: {sum(p.numel() for p in chemberta.parameters()):,}')
print(f'  Hidden size: {chemberta.config.hidden_size}')

In [ ]:
# ── Cell 6: Extract CLS embeddings (batch processing) ─────────────────────
def get_embeddings(smiles_list, tokenizer, model, batch_size=32, device=DEVICE):
    """
    Extract 768-d [CLS] embeddings from ChemBERTa for a list of SMILES.
    The [CLS] token captures the global molecular representation.
    """
    all_embeddings = []
    model.eval()

    for i in range(0, len(smiles_list), batch_size):
        batch = smiles_list[i : i + batch_size]

        # Tokenise: truncate long SMILES at 512 tokens
        encoding = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=512,
            return_tensors='pt'
        ).to(device)

        with torch.no_grad():
            outputs = model(**encoding)
            # [CLS] token is the first token — shape: (batch, hidden)
            cls_emb = outputs.last_hidden_state[:, 0, :]
            all_embeddings.append(cls_emb.cpu().numpy())

        if (i // batch_size) % 10 == 0:
            print(f'  Batch {i//batch_size + 1}/{len(smiles_list)//batch_size + 1}', end='\r')

    return np.vstack(all_embeddings)


print('Extracting ChemBERTa embeddings (frozen — no gradient)...')
print('This creates a 768-dimensional chemical fingerprint for each molecule.')
print()

emb_train = get_embeddings(smiles_train, tokenizer, chemberta)
emb_test  = get_embeddings(smiles_test,  tokenizer, chemberta)

print(f'\n✓ Train embeddings: {emb_train.shape}')
print(f'  Test  embeddings: {emb_test.shape}')

# Save embeddings — reuse without re-running if kernel restarts
np.save('embeddings_chemberta_train.npy', emb_train)
np.save('embeddings_chemberta_test.npy',  emb_test)
print('✓ Embeddings saved to disk.')

In [ ]:
# ── Cell 7: Approach A1 — ChemBERTa embeddings + RF (frozen baseline) ─────
#
# This tests whether ChemBERTa representations are more informative than
# our hand-crafted 2048-bit Morgan fingerprints alone.
# We also test a COMBINED feature vector: ChemBERTa + Morgan FP.

from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler

print('=' * 65)
print('APPROACH A1: ChemBERTa embeddings (frozen) + RF regressor')
print('=' * 65)

# Scale embeddings (RF doesn't need it, but helps with combined features)
scaler_emb = StandardScaler()
emb_train_scaled = scaler_emb.fit_transform(emb_train)
emb_test_scaled  = scaler_emb.transform(emb_test)

# Combined: ChemBERTa + Morgan fingerprints
X_combined_train = np.hstack([emb_train_scaled, X_train_fp])
X_combined_test  = np.hstack([emb_test_scaled,  X_test_fp])

results_a1 = {}

for feature_set, X_tr, X_te, label in [
    ('Morgan only (baseline)',           X_train_fp,        X_test_fp,        'Morgan'),
    ('ChemBERTa only (frozen)',          emb_train_scaled,  emb_test_scaled,  'ChemBERTa'),
    ('ChemBERTa + Morgan (combined)',    X_combined_train,  X_combined_test,  'Combined'),
]:
    print(f'\n  Feature set: {feature_set}')
    prop_results = {}

    for prop in ['HOMO', 'LUMO', 'Eg']:
        rf = RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1)
        rf.fit(X_tr, y_train[prop])
        preds = rf.predict(X_te)

        r2  = r2_score(y_test[prop], preds)
        mae = mean_absolute_error(y_test[prop], preds)
        prop_results[prop] = {'r2': r2, 'mae': mae, 'preds': preds}
        print(f'    {prop:6s}: R²={r2:.4f}  MAE={mae:.4f} eV')

    results_a1[label] = prop_results

print('\n✓ Approach A1 complete.')

In [ ]:
# ── Cell 8: Approach A2 — ChemBERTa fine-tuning (end-to-end) ──────────────
#
# Architecture:
#   ChemBERTa (last 2 layers unfrozen) → [CLS] 768-d
#       → Linear(768, 256) → GELU → Dropout(0.2)
#       → Linear(256, 64)  → GELU → Dropout(0.1)
#       → Linear(64, 1)    (separate head per property)
#
# Training strategy:
#   - Freeze first 10 transformer layers, train last 2 + regression head
#   - Low learning rate (2e-5) for transformer layers, higher (1e-3) for head
#   - Cosine LR schedule, early stopping on val loss

class MoleculeDataset(Dataset):
    def __init__(self, smiles_list, labels, tokenizer, max_len=512):
        self.smiles   = smiles_list
        self.labels   = torch.tensor(labels, dtype=torch.float32)
        self.tokenizer = tokenizer
        self.max_len  = max_len

    def __len__(self):
        return len(self.smiles)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.smiles[idx],
            truncation=True,
            max_length=self.max_len,
            return_tensors='pt',
            padding='max_length'
        )
        return {
            'input_ids':      enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'label':          self.labels[idx]
        }


class ChemBERTaRegressor(nn.Module):
    """
    ChemBERTa backbone + regression head.
    Freezes first N transformer layers, trains the rest.
    """
    def __init__(self, backbone, hidden_size=768, freeze_layers=10, dropout=0.2):
        super().__init__()
        self.backbone = backbone

        # Freeze embedding + first `freeze_layers` encoder layers
        for param in self.backbone.embeddings.parameters():
            param.requires_grad = False
        for layer in self.backbone.encoder.layer[:freeze_layers]:
            for param in layer.parameters():
                param.requires_grad = False

        # Regression head
        self.head = nn.Sequential(
            nn.Linear(hidden_size, 256),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(256, 64),
            nn.GELU(),
            nn.Dropout(dropout / 2),
            nn.Linear(64, 1)
        )

        # Initialise head weights
        for m in self.head.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, input_ids, attention_mask):
        out = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0, :]   # [CLS] token
        return self.head(cls).squeeze(-1)


def train_one_property(prop_name, y_tr, y_te, tokenizer, backbone_name,
                        epochs=30, batch_size=16, freeze_layers=10, lr_backbone=2e-5, lr_head=1e-3):
    """
    Fine-tune ChemBERTa for one property. Returns best test R² and predictions.
    """
    print(f'\n  Training for {prop_name}...')

    # Scale targets: stabilises training significantly
    scaler_y = StandardScaler()
    y_tr_scaled = scaler_y.fit_transform(y_tr.reshape(-1, 1)).ravel().astype(np.float32)

    # 90/10 internal train/val split (validation for early stopping)
    smi_t, smi_v, y_t, y_v = train_test_split(
        smiles_train, y_tr_scaled, test_size=0.1, random_state=42
    )

    train_ds = MoleculeDataset(smi_t, y_t, tokenizer)
    val_ds   = MoleculeDataset(smi_v, y_v, tokenizer)
    test_ds  = MoleculeDataset(smiles_test, y_te, tokenizer)

    train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True,  num_workers=2, pin_memory=True)
    val_dl   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
    test_dl  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

    # Fresh backbone per property (prevents cross-property contamination)
    backbone = AutoModel.from_pretrained(backbone_name)
    model    = ChemBERTaRegressor(backbone, freeze_layers=freeze_layers).to(DEVICE)

    # Separate learning rates: backbone (unfrozen layers) vs head
    backbone_params = [p for n, p in model.backbone.named_parameters() if p.requires_grad]
    head_params     = list(model.head.parameters())
    optimizer = AdamW([
        {'params': backbone_params, 'lr': lr_backbone, 'weight_decay': 0.01},
        {'params': head_params,     'lr': lr_head,     'weight_decay': 1e-4},
    ])
    scheduler = CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = nn.HuberLoss(delta=1.0)   # robust to outliers

    best_val_loss = float('inf')
    best_state    = None
    patience, no_improve = 7, 0
    train_losses, val_losses = [], []

    for epoch in range(epochs):
        # ── Train ────────────────────────────────────────────────────────
        model.train()
        train_loss = 0.0
        for batch in train_dl:
            optimizer.zero_grad()
            preds = model(batch['input_ids'].to(DEVICE),
                          batch['attention_mask'].to(DEVICE))
            loss  = criterion(preds, batch['label'].to(DEVICE))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            train_loss += loss.item()
        train_loss /= len(train_dl)

        # ── Validate ─────────────────────────────────────────────────────
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for batch in val_dl:
                preds = model(batch['input_ids'].to(DEVICE),
                              batch['attention_mask'].to(DEVICE))
                val_loss += criterion(preds, batch['label'].to(DEVICE)).item()
        val_loss /= len(val_dl)

        train_losses.append(train_loss)
        val_losses.append(val_loss)
        scheduler.step()

        if epoch % 5 == 0 or epoch == epochs - 1:
            print(f'    Epoch {epoch+1:3d}/{epochs} | train_loss={train_loss:.4f} | val_loss={val_loss:.4f}')

        # Early stopping
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state    = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            no_improve    = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f'    Early stopping at epoch {epoch+1}')
                break

    # ── Evaluate best checkpoint on test set ─────────────────────────────
    model.load_state_dict(best_state)
    model.eval()
    all_preds = []
    with torch.no_grad():
        for batch in test_dl:
            p = model(batch['input_ids'].to(DEVICE),
                      batch['attention_mask'].to(DEVICE))
            all_preds.append(p.cpu().numpy())

    preds_scaled = np.concatenate(all_preds)
    # Inverse-transform predictions back to original eV scale
    preds_ev = scaler_y.inverse_transform(preds_scaled.reshape(-1, 1)).ravel()

    r2  = r2_score(y_te, preds_ev)
    mae = mean_absolute_error(y_te, preds_ev)
    print(f'    ✓ {prop_name} test: R²={r2:.4f}  MAE={mae:.4f} eV')

    return {
        'r2':           r2,
        'mae':          mae,
        'preds':        preds_ev,
        'train_losses': train_losses,
        'val_losses':   val_losses,
        'model_state':  best_state,
        'scaler_y':     scaler_y,
    }


print('=' * 65)
print('APPROACH A2: ChemBERTa fine-tuning (last 2 layers unfrozen)')
print('=' * 65)
print('This trains for 30 epochs per property with early stopping.')
print('Expected runtime: ~8–15 min on T4 GPU, ~40 min on CPU.')
print()

results_a2 = {}
for prop, y_tr, y_te in [
    ('HOMO', y_train['HOMO'], y_test['HOMO']),
    ('LUMO', y_train['LUMO'], y_test['LUMO']),
    ('Eg',   y_train['Eg'],   y_test['Eg']),
]:
    results_a2[prop] = train_one_property(
        prop, y_tr, y_te,
        tokenizer, MODEL_NAME,
        epochs=30, batch_size=16,
        freeze_layers=10,       # Freeze first 10 of 12 layers
        lr_backbone=2e-5,       # Low LR for transformer weights
        lr_head=1e-3,           # Higher LR for fresh regression head
    )

print('\n✓ Fine-tuning complete.')

In [ ]:
# ── Cell 9: Approach A2b — Progressive unfreezing (advanced) ──────────────
#
# Instead of unfreezing a fixed number of layers upfront, we:
#   1. Train head only for 5 epochs
#   2. Unfreeze layer 11 for 5 epochs
#   3. Unfreeze layers 10–11 for remaining epochs
#
# This prevents the transformer weights from being corrupted by a
# randomly-initialised head gradient early in training.

def progressive_finetune(prop_name, y_tr, y_te, tokenizer, backbone_name,
                          epochs_per_stage=(8, 8, 14), batch_size=16,
                          lr_head=1e-3, lr_backbone=2e-5):
    print(f'\n  Progressive fine-tuning for {prop_name}...')

    scaler_y = StandardScaler()
    y_tr_scaled = scaler_y.fit_transform(y_tr.reshape(-1, 1)).ravel().astype(np.float32)

    smi_t, smi_v, y_t, y_v = train_test_split(
        smiles_train, y_tr_scaled, test_size=0.1, random_state=42
    )
    train_ds = MoleculeDataset(smi_t, y_t, tokenizer)
    val_ds   = MoleculeDataset(smi_v, y_v, tokenizer)
    test_ds  = MoleculeDataset(smiles_test, y_te, tokenizer)

    train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True,  num_workers=2, pin_memory=True)
    val_dl   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
    test_dl  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

    backbone = AutoModel.from_pretrained(backbone_name)
    model    = ChemBERTaRegressor(backbone, freeze_layers=12).to(DEVICE)  # all layers frozen initially

    criterion = nn.HuberLoss(delta=1.0)
    best_val_loss = float('inf')
    best_state    = None
    all_train_losses, all_val_losses = [], []

    # Stage 1: head only
    # Stage 2: unfreeze last 1 layer
    # Stage 3: unfreeze last 2 layers
    unfreeze_schedule = [12, 11, 10]  # layers frozen AT START of each stage

    for stage, (n_epochs, n_freeze) in enumerate(zip(epochs_per_stage, unfreeze_schedule)):
        # Re-apply freeze schedule
        for i, layer in enumerate(model.backbone.encoder.layer):
            frozen = i < n_freeze
            for p in layer.parameters():
                p.requires_grad = not frozen

        backbone_params = [p for p in model.backbone.parameters() if p.requires_grad]
        head_params     = list(model.head.parameters())

        optimizer = AdamW([
            {'params': backbone_params, 'lr': lr_backbone, 'weight_decay': 0.01},
            {'params': head_params,     'lr': lr_head,     'weight_decay': 1e-4},
        ])
        scheduler = CosineAnnealingLR(optimizer, T_max=n_epochs)

        n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f'    Stage {stage+1}: {n_epochs} epochs | unfrozen params={n_trainable:,}')

        for epoch in range(n_epochs):
            model.train()
            train_loss = 0.0
            for batch in train_dl:
                optimizer.zero_grad()
                preds = model(batch['input_ids'].to(DEVICE), batch['attention_mask'].to(DEVICE))
                loss  = criterion(preds, batch['label'].to(DEVICE))
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                train_loss += loss.item()
            train_loss /= len(train_dl)

            model.eval()
            val_loss = 0.0
            with torch.no_grad():
                for batch in val_dl:
                    preds = model(batch['input_ids'].to(DEVICE), batch['attention_mask'].to(DEVICE))
                    val_loss += criterion(preds, batch['label'].to(DEVICE)).item()
            val_loss /= len(val_dl)

            all_train_losses.append(train_loss)
            all_val_losses.append(val_loss)
            scheduler.step()

            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_state    = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    # Test set evaluation
    model.load_state_dict(best_state)
    model.eval()
    all_preds = []
    with torch.no_grad():
        for batch in test_dl:
            p = model(batch['input_ids'].to(DEVICE), batch['attention_mask'].to(DEVICE))
            all_preds.append(p.cpu().numpy())

    preds_ev = scaler_y.inverse_transform(np.concatenate(all_preds).reshape(-1, 1)).ravel()
    r2  = r2_score(y_te, preds_ev)
    mae = mean_absolute_error(y_te, preds_ev)
    print(f'    ✓ {prop_name} test: R²={r2:.4f}  MAE={mae:.4f} eV')

    return {'r2': r2, 'mae': mae, 'preds': preds_ev,
            'train_losses': all_train_losses, 'val_losses': all_val_losses}


print('=' * 65)
print('APPROACH A2b: Progressive unfreezing (3 stages)')
print('=' * 65)

results_a2b = {}
for prop, y_tr, y_te in [
    ('HOMO', y_train['HOMO'], y_test['HOMO']),
    ('LUMO', y_train['LUMO'], y_test['LUMO']),
    ('Eg',   y_train['Eg'],   y_test['Eg']),
]:
    results_a2b[prop] = progressive_finetune(
        prop, y_tr, y_te, tokenizer, MODEL_NAME,
        epochs_per_stage=(8, 8, 14), batch_size=16
    )

print('\n✓ Progressive fine-tuning complete.')

In [ ]:
# ── Cell 10: Results summary & comparison table ────────────────────────────
print('=' * 75)
print('FULL RESULTS COMPARISON')
print('=' * 75)

# RF baseline (retrain if models not loaded)
if rf_models is None:
    print('Retraining RF baseline...')
    rf_models = {}
    for prop in ['HOMO', 'LUMO', 'Eg']:
        rf = RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1)
        rf.fit(X_train_fp, y_train[prop])
        rf_models[prop] = rf

rf_results = {}
for prop in ['HOMO', 'LUMO', 'Eg']:
    preds = rf_models[prop].predict(X_test_fp)
    rf_results[prop] = {
        'r2':  r2_score(y_test[prop], preds),
        'mae': mean_absolute_error(y_test[prop], preds)
    }

print(f'\n{"Model":<45} {"HOMO R²":>9} {"LUMO R²":>9} {"Eg R²":>9} {"Avg R²":>9}')
print('-' * 83)

all_results = [
    ('RF + Morgan FP (Step 3 baseline)',    rf_results),
    ('RF + ChemBERTa frozen',               {p: results_a1['ChemBERTa'][p] for p in ['HOMO','LUMO','Eg']}),
    ('RF + ChemBERTa + Morgan (combined)',  {p: results_a1['Combined'][p]   for p in ['HOMO','LUMO','Eg']}),
    ('ChemBERTa fine-tuned (A2)',           results_a2),
    ('ChemBERTa progressive unfreeze (A2b)',results_a2b),
]

best_per_prop = {'HOMO': -999, 'LUMO': -999, 'Eg': -999}

for name, res in all_results:
    r2s  = {p: res[p]['r2']  for p in ['HOMO','LUMO','Eg']}
    avg  = np.mean(list(r2s.values()))
    for p, v in r2s.items():
        best_per_prop[p] = max(best_per_prop[p], v)
    print(f'{name:<45} {r2s["HOMO"]:>9.4f} {r2s["LUMO"]:>9.4f} {r2s["Eg"]:>9.4f} {avg:>9.4f}')

print('-' * 83)
print(f'{"Best per property":45} {best_per_prop["HOMO"]:>9.4f} {best_per_prop["LUMO"]:>9.4f} {best_per_prop["Eg"]:>9.4f}')

# Delta vs RF baseline
print(f'\n{"Improvement over RF baseline"}')
print('-' * 60)
for name, res in all_results[1:]:
    deltas = [res[p]['r2'] - rf_results[p]['r2'] for p in ['HOMO','LUMO','Eg']]
    avg_d  = np.mean(deltas)
    sym    = '+' if avg_d >= 0 else ''
    print(f'  {name:<42} avg Δ={sym}{avg_d:+.4f}')

In [ ]:
# ── Cell 11: Visualisation ─────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 11))
fig.suptitle('Transfer Learning vs RF Baseline — Predicted vs Actual',
             fontsize=15, fontweight='bold')

prop_labels = {'HOMO': 'HOMO (eV)', 'LUMO': 'LUMO (eV)', 'Eg': 'Optical Bandgap (eV)'}
prop_colors = {'HOMO': '#2980b9', 'LUMO': '#e74c3c', 'Eg': '#27ae60'}

# Row 0: best transfer learning model per property
# Row 1: RF baseline
for col, prop in enumerate(['HOMO', 'LUMO', 'Eg']):
    # Pick best TL result
    best_tl = max([results_a2, results_a2b], key=lambda r: r[prop]['r2'])
    best_name = 'Fine-tuned' if best_tl is results_a2 else 'Progressive'

    for row, (label, preds) in enumerate([
        (f'ChemBERTa {best_name}\nR²={best_tl[prop]["r2"]:.4f}  MAE={best_tl[prop]["mae"]:.4f} eV',
         best_tl[prop]['preds']),
        (f'RF Baseline\nR²={rf_results[prop]["r2"]:.4f}  MAE={rf_results[prop]["mae"]:.4f} eV',
         rf_models[prop].predict(X_test_fp)),
    ]):
        ax = axes[row, col]
        y_t = y_test[prop]

        ax.scatter(y_t, preds, alpha=0.45, s=22,
                   color=prop_colors[prop], edgecolors='white', linewidth=0.3)
        lims = [min(y_t.min(), preds.min()), max(y_t.max(), preds.max())]
        ax.plot(lims, lims, 'k--', lw=1.5, alpha=0.7)
        ax.set_xlabel(f'Actual {prop_labels[prop]}')
        ax.set_ylabel(f'Predicted {prop_labels[prop]}')
        ax.set_title(f'{prop_labels[prop]}\n{label}', fontsize=10)

plt.tight_layout()
plt.savefig('transfer_learning_predictions.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Prediction plots saved.')

# ── Training curves ──────────────────────────────────────────────────────
fig2, axes2 = plt.subplots(1, 3, figsize=(18, 5))
fig2.suptitle('ChemBERTa Training Curves (Fine-tuned)', fontsize=14, fontweight='bold')

for ax, prop in zip(axes2, ['HOMO', 'LUMO', 'Eg']):
    res = results_a2[prop]
    epochs_x = range(1, len(res['train_losses']) + 1)
    ax.plot(epochs_x, res['train_losses'], label='Train loss', color='steelblue')
    ax.plot(epochs_x, res['val_losses'],   label='Val loss',   color='coral',   linestyle='--')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Huber Loss')
    ax.set_title(f'{prop} — Training Curve')
    ax.legend()

plt.tight_layout()
plt.savefig('transfer_learning_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Training curves saved.')

In [ ]:
# ── Cell 12: Scaffold split evaluation (honest test) ──────────────────────
#
# The scaffold split is the HONEST test — if transfer learning is truly
# learning chemistry rather than memorising scaffolds, it should show a
# smaller gap between random split and scaffold split R² than RF does.

from rdkit import Chem
from rdkit.Chem.Scaffolds import MurckoScaffold
from collections import defaultdict

print('Building scaffold split for honest evaluation...')

scaffolds = defaultdict(list)
for i, smiles in enumerate(smiles_list):
    mol = Chem.MolFromSmiles(str(smiles))
    if mol:
        try:
            scaffold = MurckoScaffold.MurckoScaffoldSmiles(mol=mol, includeChirality=False)
            scaffolds[scaffold].append(i)
        except:
            scaffolds['generic'].append(i)

scaffold_sets   = sorted(scaffolds.values(), key=len, reverse=False)
target_test_n   = int(0.2 * len(df))
test_idx_sc, train_idx_sc = [], []
for group in scaffold_sets:
    if len(test_idx_sc) < target_test_n:
        test_idx_sc.extend(group)
    else:
        train_idx_sc.extend(group)

print(f'Scaffold split: {len(train_idx_sc)} train | {len(test_idx_sc)} test')

smiles_train_sc = [smiles_list[i] for i in train_idx_sc]
smiles_test_sc  = [smiles_list[i] for i in test_idx_sc]

# ChemBERTa embeddings on scaffold split
print('Extracting embeddings for scaffold split...')
emb_train_sc = get_embeddings(smiles_train_sc, tokenizer, chemberta)
emb_test_sc  = get_embeddings(smiles_test_sc,  tokenizer, chemberta)
scaler_sc    = StandardScaler()
emb_tr_sc_s  = scaler_sc.fit_transform(emb_train_sc)
emb_te_sc_s  = scaler_sc.transform(emb_test_sc)

X_train_fp_sc = X_morgan[train_idx_sc]
X_test_fp_sc  = X_morgan[test_idx_sc]
X_comb_tr_sc  = np.hstack([emb_tr_sc_s, X_train_fp_sc])
X_comb_te_sc  = np.hstack([emb_te_sc_s, X_test_fp_sc])

print('\nScaffold split R² comparison:')
print(f'{"Model":<40} {"HOMO R²":>9} {"LUMO R²":>9} {"Eg R²":>9}')
print('-' * 70)

for label, X_tr, X_te in [
    ('RF + Morgan (random split baseline)',    X_train_fp,   X_test_fp),
    ('RF + Morgan (scaffold split)',           X_train_fp_sc, X_test_fp_sc),
    ('RF + ChemBERTa+Morgan (scaffold split)', X_comb_tr_sc, X_comb_te_sc),
]:
    is_sc  = 'scaffold' in label
    y_trs  = {p: targets[p][train_idx_sc if is_sc else idx_train] for p in ['HOMO','LUMO','Eg']}
    y_tes  = {p: targets[p][test_idx_sc  if is_sc else idx_test]  for p in ['HOMO','LUMO','Eg']}

    r2s = {}
    for prop in ['HOMO', 'LUMO', 'Eg']:
        rf = RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1)
        rf.fit(X_tr, y_trs[prop])
        r2s[prop] = r2_score(y_tes[prop], rf.predict(X_te))

    print(f'{label:<40} {r2s["HOMO"]:>9.4f} {r2s["LUMO"]:>9.4f} {r2s["Eg"]:>9.4f}')

print()
print('Key question: Is the scaffold gap smaller with ChemBERTa+Morgan than with Morgan alone?')
print('If yes → transfer learning genuinely improves generalisation to unseen scaffolds.')

In [ ]:
# ── Cell 13: Save outputs ─────────────────────────────────────────────────
import pickle

# Save all transfer learning results
tl_summary = {
    'a1_rf_frozen':     results_a1,
    'a2_finetuned':     {p: {'r2': results_a2[p]['r2'], 'mae': results_a2[p]['mae']} for p in results_a2},
    'a2b_progressive':  {p: {'r2': results_a2b[p]['r2'], 'mae': results_a2b[p]['mae']} for p in results_a2b},
    'rf_baseline':      rf_results,
}

with open('transfer_learning_results.pkl', 'wb') as f:
    pickle.dump(tl_summary, f)

# Save model states
torch.save({p: results_a2[p]['model_state']  for p in results_a2},  'chemberta_finetuned.pt')
torch.save({p: results_a2b[p]                for p in results_a2b}, 'chemberta_progressive.pt')

# Save embeddings
np.save('embeddings_chemberta_train.npy', emb_train)
np.save('embeddings_chemberta_test.npy',  emb_test)

print('Files saved:')
print('  transfer_learning_results.pkl  — R²/MAE for all models')
print('  chemberta_finetuned.pt          — A2 model weights')
print('  chemberta_progressive.pt        — A2b model weights')
print('  embeddings_chemberta_train.npy  — 768-d train embeddings')
print('  embeddings_chemberta_test.npy   — 768-d test  embeddings')
print('  transfer_learning_predictions.png')
print('  transfer_learning_curves.png')

try:
    from google.colab import files
    for fname in [
        'transfer_learning_results.pkl',
        'transfer_learning_predictions.png',
        'transfer_learning_curves.png',
    ]:
        files.download(fname)
except Exception:
    pass

print('\n✓ STEP 7 COMPLETE')

---
## Interpretation Guide

### Reading your results

| Outcome | What it means | What to do |
|---------|---------------|------------|
| **ChemBERTa (frozen) > RF** | ChemBERTa representations encode more chemistry than Morgan FP | Use combined embeddings in production |
| **Fine-tuned > Frozen** | The HOMO/LUMO/Eg task benefits from adapting the transformer weights | Use fine-tuned model |
| **Combined (Emb + Morgan) best** | Both representation types are complementary | This is the recommended production model |
| **Scaffold gap narrowed vs RF** | Transfer learning genuinely generalises — not just memorising | Strong positive result |
| **Scaffold gap unchanged** | The bottleneck is still data scarcity (need more diverse molecules) | Collect more data, try larger pretrained model |

### Why this approach is appropriate for your dataset size

ChemBERTa was pretrained on **100,000 SMILES** — 64× more than your 1,571 molecules.  
The frozen-embedding approach (Cell 7) is essentially free — you get a richer 768-d representation  
without risking overfitting, because no new parameters are being trained on your small dataset.  
Fine-tuning (Cell 8) is higher risk but higher reward — the early stopping and layer freezing  
protect against the gradient explosion that killed your Step 3 neural network.

### If results are still below RF

This would mean the domain gap between ZINC (drug-like molecules) and your  
organic photovoltaic acceptors is too large for ChemBERTa to bridge.  
Next options:
- Try `seyonec/PubChem10M_SMILES_BPE_450k` — pretrained on 10M molecules, broader coverage
- Try `DeepChem/ChemBERTa-10M-MLM` — even larger pretraining set
- Consider GROVER or Uni-Mol which include quantum chemistry properties in pretraining